In [ ]:
!pip install -q -U transformers peft accelerate bitsandbytes trl datasets wandb

In [ ]:
from kaggle_secrets import UserSecretsClient
import os

user_secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = user_secrets.get_secret("HF_TOKEN")
os.environ["WANDB_API_KEY"] = user_secrets.get_secret("WANDB_API_KEY")
print("Secrets loaded!")

In [ ]:
%%writefile /kaggle/working/dpo_train1.py

import os
import torch
import wandb
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from trl import DPOTrainer, DPOConfig
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

user_secrets = UserSecretsClient()
login(token=user_secrets.get_secret("HF_TOKEN"))
wandb.login(key=user_secrets.get_secret("WANDB_API_KEY"))

base_model_id = "meta-llama/Llama-3.2-1B-Instruct"
adapter_id = "pranav6905/Llama-3.2-1B-SFT-DPOMix-Adapters"
max_seq_length = 512

tokenizer = AutoTokenizer.from_pretrained(base_model_id)
if "<|finetune_right_pad_id|>" in tokenizer.get_vocab():
    tokenizer.pad_token = "<|finetune_right_pad_id|>"
else:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype
)

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config=bnb_config,
    device_map={"": int(os.environ.get("LOCAL_RANK", 0))},  # each process gets its GPU
    attn_implementation="eager"
)

model = PeftModel.from_pretrained(base_model, adapter_id, is_trainable=True)

dataset = load_dataset("argilla/dpo-mix-7k", split="train")

def format_dpo_data(example):
    prompt_messages = example["chosen"][:-1]
    return {
        "prompt": tokenizer.apply_chat_template(prompt_messages, tokenize=False, add_generation_prompt=True),
        "chosen": tokenizer.apply_chat_template(example["chosen"], tokenize=False),
        "rejected": tokenizer.apply_chat_template(example["rejected"], tokenize=False),
    }

dpo_dataset = dataset.map(format_dpo_data, remove_columns=dataset.column_names)

trainer = DPOTrainer(
    model=model,
    train_dataset=dpo_dataset,
    processing_class=tokenizer,
    args=DPOConfig(
        beta=0.1,
        max_length=max_seq_length,
        dataset_num_proc=4,
        output_dir="./Llama-DPO-Output",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        gradient_checkpointing=True,
        max_grad_norm=0.3,
        learning_rate=1e-5,
        warmup_steps=20,
        num_train_epochs=1,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        ddp_find_unused_parameters=False,
        logging_steps=10,
        optim="paged_adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=3407,
        report_to="wandb",
        run_name="llama-1b-dpo-phase2",
    ),
)

trainer.train()

trainer.model.save_pretrained("/kaggle/working/Llama-DPO-Local-Save")
tokenizer.save_pretrained("/kaggle/working/Llama-DPO-Local-Save")
trainer.model.push_to_hub("pranav6905/Llama-3.2-1B-DPO-DPOMix-Adapters", token=user_secrets.get_secret("HF_TOKEN"))
tokenizer.push_to_hub("pranav6905/Llama-3.2-1B-DPO-DPOMix-Adapters", token=user_secrets.get_secret("HF_TOKEN"))
print("Done!")

In [ ]:
%%bash
torchrun --nproc_per_node=2 /kaggle/working/dpo_train1.py 2>&1 | tee /kaggle/working/training_log1.txt